## Classification with SCRDR

ဒီ notebook မှာတော့ interactive ပုံစံမျိုးနဲ့ လူကိုယ်တိုင်က rule တွေကို manual သတ်မှတ်ပေးတာမျိုး မဟုတ်ပဲ၊ ဒေတာကို လေ့လာပြီး automatic rule induction လုပ်တဲ့နည်းကို မိတ်ဆက်ပေးချင်ပါတယ်။ လက်တွေ့မှာ ဒီနည်းလမ်းမျိုးကို သေချာပြင်ဆင်ထားတဲ့ ဒေတာတွေရှိရင် အသုံးချလို့ ရနိုင်ပါတယ်။ Coding ကတော့ လုပ်ဖို့လိုအပ်ပေမဲ့ လက်ရှိအချိန်မှာဆိုရင်တော့ vibe coding လည်း လုပ်လို့ ရတာမို့ အများကြီး မြန်မြန်ဆန်ဆန် rule induction လုပ်ဖို့ ဖြစ်နိုင်ပါတယ်။ သို့သော် ဆရာ အမြဲပြောနေသလိုပါပဲ LLM နဲ့ ရေးပြီးထွက်လာတဲ့ code ကိုတော့ ပြန်ဖတ်ပါ၊ ပြန်စစ်ပါ။ အနည်းဆုံးတော့ experiment result တွေ၊ evaluation score တွေထုတ်ကြည့်ပြီးတော့ confirmation လုပ်မှ ဖြစ်လိမ့်မယ်။  

In [1]:
%pwd

'/home/ye/exp/scrdr_tuto'

In [2]:
%cd ./classi

/home/ye/exp/scrdr_tuto/classi


In [3]:
!ls *.py

scrdr_learner.py


## scrdr_learner.py Code

In [4]:
!cat ./scrdr_learner.py

## For understanding automatic learning of SCRDR rules to my AI Engineering (Fundamental) Class students
## Written by Ye Kyaw Thu, LU Lab., Myanmar
## Last Updated: 12 April 2026

import pandas as pd
import json
import argparse
import numpy as np
import os
import re
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder

class RDRNode:
    def __init__(self, condition=None, conclusion=None, cornerstone=None):
        self.condition = condition
        self.conclusion = self._normalize_label(conclusion)
        self.cornerstone = cornerstone
        self.if_true = None  
        self.if_false = None 

    def _normalize_label(self, label):
        if label is None: return None
        s = str(label).strip()
        if s.endswith('.0'): s = s[:-2]
        return s

    def evaluate(self, row):
     

## --help

ဘယ်လို commandline argument တွေရှိသလဲ ဆိုတာကို --help option ပေးပြီး run ကြည့်ရအောင်။  

In [6]:
!python ./scrdr_learner.py --help

usage: scrdr_learner.py [-h] --input INPUT --target TARGET
                        [--exclude [EXCLUDE ...]] [--output OUTPUT]
                        [--plot PLOT]

Universal Auto-RDR Learner

options:
  -h, --help            show this help message and exit
  --input INPUT         CSV or Excel path
  --target TARGET       Target column
  --exclude [EXCLUDE ...]
                        Columns to exclude
  --output OUTPUT       JSON model output
  --plot PLOT           Confusion matrix PNG output


## Let's See Shell Script

SCRDR rule induction, classification လုပ်ဖို့အတွက် ပြင်ဆင်ခဲ့တဲ့ shell script က အောက်ပါအတိုင်းပါ။  

In [5]:
!cat ./run_scrdr.sh

#!/bin/bash

mkdir results_scrdr;

time python ./scrdr_learner.py --input ./data/iris.data --target class \
--plot ./results_scrdr/iris_rdr_cf.png --output  ./results_scrdr/iris_rdr_model.json

time python ./scrdr_learner.py --input ./data/titanic1.csv --target Survived \
--exclude PassengerId --plot  ./results_scrdr/titanic_rdr_cf.png \
--output  ./results_scrdr/titanic_rdr_model.json

time python ./scrdr_learner.py --input ./data/wine.data --target Class \
--plot  ./results_scrdr/wine_rdr_cf.png --output  ./results_scrdr/wine_rdr_model.json

time python ./scrdr_learner.py --input ./data/agaricus-lepiota.data --target class \
--plot  ./results_scrdr/mushroom_rdr_cf.png --output  ./results_scrdr/mushroom_rdr_model.json

time python ./scrdr_learner.py --input ./data/wdbc.data --target Diagnosis \
--exclude "ID" --plot  ./results_scrdr/breast_cancer_rdr_cf.png \
--output  ./results_scrdr/breast_cancer_rdr_model.json



## Let's Run

Shell script ကို run ကြည့်ကြရအောင်။ Rule induction လုပ်ပြီး SCRDR tree ဆောက်မယ် ပြီးတော့ classification လုပ်ပြီး ရလာတဲ့ ရလဒ်တွေကို မြင်ရပါလိမ့်မယ်။ Run time က မြန်ပါတယ်။  

In [7]:
!./run_scrdr.sh

mkdir: cannot create directory ‘results_scrdr’: File exists
Building RDR tree on 120 samples...

--- CLASSIFICATION REPORT ---
                 precision    recall  f1-score   support

    Iris-setosa       1.00      1.00      1.00        10
Iris-versicolor       1.00      0.89      0.94         9
 Iris-virginica       0.92      1.00      0.96        11

       accuracy                           0.97        30
      macro avg       0.97      0.96      0.97        30
   weighted avg       0.97      0.97      0.97        30

Confusion matrix saved to ./results_scrdr/iris_rdr_cf.png

real	0m1.078s
user	0m6.052s
sys	0m0.087s
Building RDR tree on 712 samples...

--- CLASSIFICATION REPORT ---
              precision    recall  f1-score   support

           0       0.82      0.89      0.85       105
           1       0.82      0.72      0.76        74

    accuracy                           0.82       179
   macro avg       0.82      0.80      0.81       179
weighted avg       0.82      0.8

## Let's Check RDR Models

RDR model တွေနဲ့ confusion matrix ဂရဖ်ပုံတွေက results_scrdr/ ဆိုတဲ့ ဖိုလ်ဒါအောက်ထဲမှာ သိမ်းပေးပါလိမ့်မယ်။  

In [8]:
!ls ./results_scrdr/

breast_cancer_rdr_cf.png      mushroom_rdr_cf.png      wine_rdr_cf.png
breast_cancer_rdr_model.json  mushroom_rdr_model.json  wine_rdr_model.json
iris_rdr_cf.png		      titanic_rdr_cf.png
iris_rdr_model.json	      titanic_rdr_model.json


In [9]:
%cd ./results_scrdr/

/home/ye/exp/scrdr_tuto/classi/results_scrdr


## Titanic Dataset

Titanic dataset ရဲ့ ထိပ်ဆုံး အကြောင်း ၃၀ ကို ရိုက်ထုတ်ကြည့်ရအောင်။  

In [10]:
!head -n 30 ../data/titanic1.csv

PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
1,0,3,male,22,1,0,A/5 21171,7.25,,S
2,1,1,female,38,1,0,PC 17599,71.2833,C85,C
3,1,3,female,26,0,0,STON/O2. 3101282,7.925,,S
4,1,1,female,35,1,0,113803,53.1,C123,S
5,0,3,male,35,0,0,373450,8.05,,S
6,0,3,male,,0,0,330877,8.4583,,Q
7,0,1,male,54,0,0,17463,51.8625,E46,S
8,0,3,male,2,3,1,349909,21.075,,S
9,1,3,female,27,0,2,347742,11.1333,,S
10,1,2,female,14,1,0,237736,30.0708,,C
11,1,3,female,4,1,1,PP 9549,16.7,G6,S
12,1,1,female,58,0,0,113783,26.55,C103,S
13,0,3,male,20,0,0,A/5. 2151,8.05,,S
14,0,3,male,39,1,5,347082,31.275,,S
15,0,3,female,14,0,0,350406,7.8542,,S
16,1,2,female,55,0,0,248706,16,,S
17,0,3,male,2,4,1,382652,29.125,,Q
18,1,2,male,,0,0,244373,13,,S
19,0,3,female,31,1,0,345763,18,,S
20,1,3,female,,0,0,2649,7.225,,C
21,0,2,male,35,0,0,239865,26,,S
22,1,2,male,34,0,0,248698,13,D56,S
23,1,3,female,15,0,0,330923,8.0292,,Q
24,1,1,male,28,0,0,113788,35.5,A6,S
25,0,3,female,8,3,1,349909,21.075,,S
26,1,3,femal

## Let's Learn SCRDR Model of Titanic

Titanic dataset အတွက် rule induction ကို ဘယ်လိုလုပ်ထားသလဲ ဆိုတာကို လေ့လာကြည့်ကြရအောင်။  

In [11]:
!cat ./titanic_rdr_model.json

{
    "rule": "Default",
    "conclusion": "0",
    "cornerstone_idx": 331,
    "exception": {
        "rule": "Ticket == 19947",
        "conclusion": "1",
        "cornerstone_idx": 55,
        "exception": null,
        "alternative": {
            "rule": "Sex == female",
            "conclusion": "1",
            "cornerstone_idx": 865,
            "exception": {
                "rule": "Ticket == 347082",
                "conclusion": "0",
                "cornerstone_idx": 542,
                "exception": null,
                "alternative": {
                    "rule": "Ticket == 315084",
                    "conclusion": "0",
                    "cornerstone_idx": 534,
                    "exception": null,
                    "alternative": {
                        "rule": "Ticket == 4133",
                        "conclusion": "0",
                        "cornerstone_idx": 409,
                        "exception": null,
                        "alternative": {
          

## ML vs SCRDR  

Machine learning နဲ့ SCRDR အကြား ရလဒ်တွေကို နှိုင်းယှဉ်ကြည့်ကြရအောင်။  

|Dataset|DT|RF|SVM|NB|LR|SCRDR|
| ---| --- | --- | --- | --- | --- | --- |
|Iris|1.00|1.00|1.00|1.00|1.00|0.97|
|Wine|0.94|1.00|1.00|1.00|1.00|0.72|
|Mushroom|1.00|1.00|1.00|0.95|1.00|1.00|
|WDBC|0.95|0.96|0.97|0.96|0.97|0.84|
|Titanic|0.78|0.80|0.82|0.78|0.80|0.82|

အခြေခံအားဖြင့် ML မော်ဒယ်တွေရဲ့ ရလဒ်တွေက သာတယ်။ သို့သော်လည်း RDR က Titanic ဒေတာအတွက်ဆိုရင် SVM လိုပဲ အကောင်းဆုံးရလဒ်ကို ထုတ်ပေးနိုင်ပါတယ်။ Mushroom ဒေတာအတွက်ဆိုရင်လည်း NB ထက်သာပြီး တခြား ML မော်ဒယ်တွေလိုပဲ အကောင်းဆုံး ရလဒ်ကို ထုတ်ပေးနိုင်ပါတယ်။ သို့သော်လည်း Wine နဲ့ WDBC (ရင်သားကင်ဆာ) ဒေတာမှာတော့ ရလဒ်က မကောင်းပါဘူး။ တစ်ခုရှိတာက RDR ရဲ့ အားသာချက်ကတော့ မော်ဒယ်က လုံးဝ transparent ဖြစ်တာမို့လို့ rule တွေကို ဝင်လေ့လာပြီး ပြင်နိုင်ရင် (သို့မဟုတ်) ဆရာ့လက်ရှိ code ကို ရဲ့ automatic rule extraction အပိုင်းကို ဝင်ပြင်နိုင်ရင် လက်ရှိနည်းနေတဲ့ ရလဒ်ထက် ပိုကောင်းအောင် လုပ်လို့ ရပါလိမ့်မယ်။  လက်တွေ့ ဒေတာက သိပ်မရှိတဲ့ အခြေအနေမျိုး၊ domain expert ကိုလည်း မေးမြန်းပြီး ဆွေးနွေးလို့ ရနိုင်တဲ့ အခြေအနေမျိုးမှာ SCRDR နည်းလမ်းက အသုံးဝင်ပါလိမ့်မယ်။ AI အင်ဂျင်နီယာဖြစ်ဖို့ ကြိုးစားနေကြတဲ့ ကျောင်းသား/သူ တွေအတွက်က လက်တွေ့ engineering thenique တစ်ခုအနေနဲ့လည်း သိထားသင့်ပါတယ်။  